# Image Color Compression via K-Means Clustering

This notebook demonstrates **Vector Quantization / Color Quantization** using K-Means clustering.

### Concepts Covered:
1. Representing 2D RGB images as 3D feature spaces $\mathbb{R}^3$.
2. Clustering pixel color vectors into $k$ optimal color centroids using `MiniBatchKMeans`.
3. Reconstructing quantized images and evaluating visual fidelity across varying $k$.
4. Inspecting the learned color palettes.
5. Computing theoretical storage compression and bitrate reduction.

### 1. Environment Setup & Library Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import MiniBatchKMeans
from skimage import data
import urllib.request
from PIL import Image
import io

# Configure plotting defaults
%matplotlib inline
plt.rcParams['figure.figsize'] = [14, 8]
plt.rcParams['figure.dpi'] = 120

### 2. Load & Prepare the Image
We convert pixel intensities from integer range `[0, 255]` to floating point `[0.0, 1.0]` and flatten the spatial matrix $(H, W, C)$ into a 2D feature matrix of shape $(N, 3)$ where $N = H \times W$.

In [ ]:
# Load standard sample image (scikit-image astronaut)
# You can also use data.chelsea(), data.coffee(), or your own URL
original_img = data.astronaut()

# Normalize pixel values to [0.0, 1.0]
img_normalized = np.array(original_img, dtype=np.float64) / 255.0
height, width, channels = img_normalized.shape

# Flatten spatial dimensions into an (N, 3) matrix
# Every row corresponds to a single pixel with features: Red, Green, Blue
pixel_matrix = img_normalized.reshape(-1, channels)

print(f"Original dimensions : {height}px height x {width}px width x {channels} channels")
print(f"Pixel matrix shape  : {pixel_matrix.shape[0]:,} total pixels (samples) with {pixel_matrix.shape[1]} color features (R, G, B)")

# Count total unique color triples in the uncompressed image
unique_colors_orig = len(np.unique(pixel_matrix, axis=0))
print(f"Unique colors       : {unique_colors_orig:,}")

### 3. Color Quantization via K-Means
We use `MiniBatchKMeans` to rapidly cluster hundreds of thousands of pixels into $k$ centroids in $\mathbb{R}^3$.

In [ ]:
def compress_image_kmeans(pixels, original_shape, k=16, random_state=42):
    """
    Fits K-Means to find k representative color centroids, replaces
    each pixel with its closest centroid, and reconstructs the image.
    """
    kmeans = MiniBatchKMeans(
        n_clusters=k,
        batch_size=4096,
        n_init=3,
        random_state=random_state
    )
    # Fit cluster centers and assign cluster labels (0 to k-1) to each pixel
    labels = kmeans.fit_predict(pixels)
    palette = kmeans.cluster_centers_
    
    # Quantize: Replace each pixel index with its 3D centroid coordinates
    compressed_pixels = palette[labels]
    
    # Reshape back to image format (H, W, 3) and clip to valid RGB ranges [0, 1]
    compressed_img = np.clip(compressed_pixels.reshape(original_shape), 0.0, 1.0)
    
    return compressed_img, palette, labels

### 4. Run Compression for Multiple $k$ Values
We evaluate compression fidelity across $k \in \{64, 16, 4, 2\}$.

In [ ]:
k_values = [64, 16, 4, 2]
compressed_results = {}

for k in k_values:
    print(f"Clustering into k = {k} color centroids...")
    comp_img, palette, labels = compress_image_kmeans(
        pixel_matrix, 
        (height, width, channels), 
        k=k
    )
    compressed_results[k] = {
        "img": comp_img,
        "palette": palette,
        "labels": labels
    }
print("Finished clustering all configurations.")

### 5. Side-by-Side Visual Comparison

In [ ]:
fig, axes = plt.subplots(1, len(k_values) + 1, figsize=(22, 5))

# Original image
axes[0].imshow(original_img)
axes[0].set_title(f"Original Image\n({unique_colors_orig:,} colors)", fontsize=12, pad=10)
axes[0].axis("off")

# Compressed images
for idx, k in enumerate(k_values, start=1):
    axes[idx].imshow(compressed_results[k]["img"])
    axes[idx].set_title(f"Quantized (k = {k})\n({k} colors)", fontsize=12, pad=10)
    axes[idx].axis("off")

plt.tight_layout()
plt.show()

### 6. Inspecting Learned Color Palettes (Centroids)
Each cluster center $\mu_i \in \mathbb{R}^3$ represents an entry in the quantized color palette.

In [ ]:
fig, axes = plt.subplots(len(k_values), 1, figsize=(10, 3.5))

for idx, k in enumerate(k_values):
    palette = compressed_results[k]["palette"]
    # Format palette into a 1 x k x 3 color swatch
    palette_strip = np.clip(palette, 0.0, 1.0)[np.newaxis, :, :]
    
    axes[idx].imshow(palette_strip, aspect='auto')
    axes[idx].set_title(f"Centroid Palette for k = {k}", fontsize=10, loc='left')
    axes[idx].set_xticks([])
    axes[idx].set_yticks([])

plt.tight_layout()
plt.show()

### 7. Theoretical Bitrate Reduction and Storage Analysis

**Formulation:**
- **Uncompressed Raw Representation:**
  $$\text{Bits}_{\text{raw}} = H \times W \times 24 \text{ bits}$$
- **Quantized Palette Representation:**
  1. **Palette Table:** $k \times 24 \text{ bits}$ (storing $k$ RGB triplets)
  2. **Index Map:** $H \times W \times \lceil\log_2(k)\rceil \text{ bits}$ (each pixel only stores its centroid index from $0$ to $k-1$)
  $$\text{Bits}_{\text{quantized}} = (k \times 24) + \left(H \times W \times \lceil\log_2(k)\rceil\right)$$

In [ ]:
def calculate_compression_stats(h, w, k):
    raw_bits = h * w * 24
    raw_kb = raw_bits / (8 * 1024)
    
    bpp = int(np.ceil(np.log2(k)))
    palette_bits = k * 24
    index_bits = h * w * bpp
    
    quantized_bits = palette_bits + index_bits
    quantized_kb = quantized_bits / (8 * 1024)
    savings_pct = (1.0 - (quantized_bits / raw_bits)) * 100.0
    
    return bpp, raw_kb, quantized_kb, savings_pct

print(f"{'k':<6} | {'Bits/Pixel':<12} | {'Raw Size (KB)':<14} | {'Quantized (KB)':<16} | {'Compression Ratio'}")
print("-" * 70)

for k in k_values:
    bpp, raw_kb, q_kb, savings = calculate_compression_stats(height, width, k)
    print(f"{k:<6} | {bpp:<12} | {raw_kb:<14.1f} | {q_kb:<16.1f} | {savings:.1f}% space saved")